## Loading the model (distilbert-base-uncased-finetuned-sst-2-english)


In [ ]:
import torch
from transformers import DistilBertTokenizer, pipeline

tokenizer = DistilBertTokenizer.from_pretrained(
    "distilbert-base-uncased-finetuned-sst-2-english"
)
model = pipeline(
    task='text-classification',
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

c:\Users\1\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\1\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\1\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Py

# Loading the data


In [ ]:
import pandas as pd

df = pd.read_csv("../data/scored_data/cleaned_reviews_all_flags_majority.csv")

## Output prediction

In [ ]:
# our mappings are 0 negative and 1 positive
def predict_sentiment(text):
    result = model.predict(text)[0]
    return 1 if result['label'] == 'POSITIVE' else 0

In [ ]:
df["transformer_prediction"] = df.apply(
    lambda x: predict_sentiment(str(x["review_title"]) + " " + str(x["review_body"])), axis=1
)

stratified_sample = df.groupby('category', group_keys=False).apply(lambda x: x.sample(5), include_groups=False)
stratified_sample[["review_title", "review_body", "transformer_prediction"]]

,review_title,review_body,transformer_prediction
192,NaN,work fine need box go connect apps volume low ...,1
200,not work consistently,work one five time take major event anyway hou...,0
249,false advertise,item not advertise project picture galaxy not ...,0
189,NaN,love picture clear26 sound good well not big f...,0
187,NaN,ability cast screen angle id like able adjust ...,1
105,NaN,vegetable oil work like need vegetable oil bud...,0
5,not happy,didnt get order pic get,0
116,NaN,cook oil product store neutral flavor use bake...,0
47,short not sweet,get pay thing say items not leave delivery per...,0
31,great quality value goto cook oil,ive buy great value vegetable oil years never ...,1


## Model vs SOTA Metrics (same test split)

This section compares:
- **Optimized model (Dept 1 output):** GaussianNB from Task 4 setup
- **SOTA model:** `distilbert-base-uncased-finetuned-sst-2-english`

Ground truth is `final_sentiment` (binary only: positive/negative), and both models are evaluated on the **same test records**.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# ---------------- PREPARE LABELS ----------------
df_eval = df[df["final_sentiment"].isin(["positive", "negative"])].copy().reset_index(drop=True)

# SOTA label mapping from pipeline output to Task labels
# POSITIVE -> positive, NEGATIVE -> negative
if "transformer_prediction" not in df_eval.columns:
    raise ValueError("Missing 'transformer_prediction'. Run the SOTA inference cell first.")

df_eval["sota_label"] = df_eval["transformer_prediction"].map({1: "positive", 0: "negative"})

# ---------------- LOAD FEATURES FOR OPTIMIZED MODEL ----------------
X_glove = pd.read_csv("../task3/text representation/glove_no_special_no_lowercase.csv")
X_glove = X_glove.select_dtypes(include=["number"]).reset_index(drop=True)

# Align lengths between labels and features
min_len = min(len(df_eval), len(X_glove))
df_eval = df_eval.iloc[:min_len].reset_index(drop=True)
X_glove = X_glove.iloc[:min_len].reset_index(drop=True)

y = df_eval["final_sentiment"]
meta = df_eval[["category", "sota_label"]]

# Same split recipe used in Task 4
X_train, X_test, y_train, y_test, meta_train, meta_test = train_test_split(
    X_glove,
    y,
    meta,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ---------------- OPTIMIZED MODEL (Dept 1) ----------------
optimized_model = GaussianNB()
optimized_model.fit(X_train, y_train)

optimized_pred = pd.Series(optimized_model.predict(X_test), index=y_test.index)
sota_pred = pd.Series(meta_test["sota_label"].values, index=y_test.index)

# ---------------- METRICS ----------------
def compute_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, pos_label="positive", zero_division=0),
        "Recall": recall_score(y_true, y_pred, pos_label="positive", zero_division=0),
        "F1 (positive)": f1_score(y_true, y_pred, pos_label="positive", zero_division=0),
        "F1 (macro)": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Confusion Matrix": confusion_matrix(y_true, y_pred).tolist()
    }

overall_results = pd.DataFrame(
    {
        "Optimized Model": compute_metrics(y_test, optimized_pred),
        "SOTA Model": compute_metrics(y_test, sota_pred)
    }
)

overall_results

In [ ]:
# Per-category breakdown (required: Accuracy + macro-F1)
per_category_rows = []

for category_name in sorted(meta_test["category"].dropna().unique()):
    mask = meta_test["category"] == category_name

    y_true_cat = y_test[mask]
    opt_cat = optimized_pred[mask]
    sota_cat = sota_pred[mask]

    if len(y_true_cat) == 0:
        continue

    per_category_rows.append(
        {
            "Category": category_name,
            "Optimized Acc": accuracy_score(y_true_cat, opt_cat),
            "Optimized F1 (macro)": f1_score(y_true_cat, opt_cat, average="macro", zero_division=0),
            "SOTA Acc": accuracy_score(y_true_cat, sota_cat),
            "SOTA F1 (macro)": f1_score(y_true_cat, sota_cat, average="macro", zero_division=0),
            "Samples": len(y_true_cat)
        }
    )

per_category_results = pd.DataFrame(per_category_rows).sort_values("Category").reset_index(drop=True)
per_category_results